In [1]:
# %pip install transformers evaluate torch peft
# %pip install accelerate

# Set up 

In [ ]:
from pathlib import Path
import sys

import pandas as pd

for candidate in [Path.cwd(), *Path.cwd().parents]:
    src_path = candidate / "src"
    if (src_path / "spotbot").exists():
        sys.path.insert(0, str(src_path))
        REPO_ROOT = candidate
        break

from spotbot.data import load_hc3_csv, load_hc3_from_huggingface
from spotbot.preprocessing import expand_answers, prepare_question_groups, remove_invalid_answers
from spotbot.splitting import (
    component_summary,
    make_group_split,
    verify_answer_text_isolation,
    verify_component_isolation,
    verify_split_isolation,
)

REPO_ROOT

PosixPath('/Users/ngokykhanhthu/2026/chewy')

In [4]:
from spotbot.prepared import load_prepared_splits


In [5]:
DATA_DIR = REPO_ROOT / "data"
SPLIT_DIR = DATA_DIR / "splits"
data_path = DATA_DIR / "hc3_all.csv"
if data_path.exists():
    raw_hc3 = load_hc3_csv(data_path)
else:
    raw_hc3 = load_hc3_from_huggingface()

schema = pd.DataFrame({
    "column": raw_hc3.columns,
    "dtype": [str(raw_hc3[col].dtype) for col in raw_hc3.columns],
})
print(f"Raw question rows: {len(raw_hc3):,}")
display(schema)
display(raw_hc3.head(3))

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Hello-SimpleAI/HC3' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Raw question rows: 24,322


,column,dtype
0,question,object
1,human_answers,object
2,chatgpt_answers,object
3,index,float64
4,source,object


,question,human_answers,chatgpt_answers,index,source
0,"Why is every book I hear about a "" NY Times # ...","[Basically there are many categories of "" Best...",[There are many different best seller lists th...,NaN,reddit_eli5
1,"If salt is so bad for cars , why do we use it ...",[salt is good for not dying in car crashes and...,[Salt is used on roads to help melt ice and sn...,NaN,reddit_eli5
2,Why do we still have SD TV channels when HD lo...,[The way it works is that old TV stations got ...,[There are a few reasons why we still have SD ...,NaN,reddit_eli5


In [6]:
frozen_train, frozen_val, frozen_test = load_prepared_splits(
    data_path if data_path.exists() else None,
    split_dir=SPLIT_DIR,
)


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Hello-SimpleAI/HC3' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


# Export to csv to train model on google collab's gpu 

In [8]:
frozen_train.to_csv('./../../data/processed/train_dataset.csv', index = False)

In [7]:
frozen_val.to_csv('./../../data/processed/val_dataset.csv', index = False)
# frozen_test.to_csv('./../../data/processed/test_dataset.csv', index = False)

In [7]:
frozen_train.head(5)

,split,question_group_id,component_id,question,source,text,label,answer_index
0,train,qg_0001a99a9cfea00d,ac_e5ab9da1516e85b9,why did gas prices triple in a decades time ? ...,reddit_eli5,A lot of it has to do with rising third world ...,0,0
1,train,qg_0001a99a9cfea00d,ac_e5ab9da1516e85b9,why did gas prices triple in a decades time ? ...,reddit_eli5,URL_0 Here 's an answer to it being relative t...,0,1
2,train,qg_0001a99a9cfea00d,ac_e5ab9da1516e85b9,why did gas prices triple in a decades time ? ...,reddit_eli5,Because we continued to pay what they are aski...,0,2
3,train,qg_0001a99a9cfea00d,ac_e5ab9da1516e85b9,why did gas prices triple in a decades time ? ...,reddit_eli5,There are several reasons why gas prices have ...,1,0
4,train,qg_000299619139c47d,ac_ee17192f440cf24a,How accurate is the statement that we are all ...,reddit_eli5,Not every element . The hydrogen on Earth coul...,0,0


In [8]:
frozen_train.shape

(65401, 8)

In [9]:
frozen_train['answer_index'].value_counts()

answer_index
0    37242
1    14329
2    13830
Name: count, dtype: int64